# Analysis Project

Now it is your turn to do an analysis yourself. You will pick one of the sources of the H.E.S.S. public data set and do a complete analysis yourself. At the end, you will have a significant detection of an object, a sky map of the surroundings of the object, and its energy spectrum.

You mainly need to copy and paste code from the Crab Nebula notebook which we have discussed in the lecture, and make the necessary changes for the new source. Please put your code in the cells starting with 

```
## your code here
```

You do not need to edit any other cells. If you want to add additional output or test some things, you can create new cells for that.

## Imports

Lets start with importing all the modules that we have learned of. There is nothing for you to do here.

In [ ]:
import os

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import numpy as np

In [ ]:
from numpy import sqrt

In [ ]:
from scipy.stats import norm

In [ ]:
import astropy.units as u

In [ ]:
from astropy.coordinates import (
    SkyCoord, 
    Angle,
)

In [ ]:
from gammapy.data import (
    DataStore,
)

In [ ]:
from gammapy.datasets import (
    SpectrumDataset,
    FluxPointsDataset,
    MapDatasetOnOff,
    MapDataset
)

In [ ]:
from gammapy.makers import (
    DatasetsMaker,
    MapDatasetMaker,
    SpectrumDatasetMaker,
    SafeMaskMaker,
    ReflectedRegionsBackgroundMaker,
    RingBackgroundMaker,
)

In [ ]:
from gammapy.estimators import (
    ExcessMapEstimator,
    FluxPointsEstimator,
    LightCurveEstimator
)

In [ ]:
from gammapy.maps import (
    Map,
    MapAxis, 
    WcsGeom,
    RegionGeom,
)

In [ ]:
from regions import CircleSkyRegion

In [ ]:
from gammapy.estimators.utils import find_peaks

In [ ]:
from gammapy.utils.check import check_tutorials_setup

In [ ]:
from gammapy.stats import WStatCountsStatistic

In [ ]:
from gammapy.visualization import plot_spectrum_datasets_off_regions

In [ ]:
import scipy.stats

In [ ]:
from scipy.stats import norm

In [ ]:
from gammapy.modeling.models import (
    Models,
    SkyModel,
    PowerLawSpectralModel,
    ExpCutoffPowerLawSpectralModel,
    PointSpatialModel,
    GaussianSpatialModel
)

In [ ]:
from gammapy.modeling.models import create_crab_spectral_model

In [ ]:
from gammapy.modeling import Fit

In [ ]:
fit = Fit()

## Check the Data

In [ ]:
if not os.path.exists('gammapy-data'):
    check_tutorials_setup()
else:
    print('Great your setup is correct!')

In [ ]:
os.environ['GAMMAPY_DATA'] = 'gammapy-data/2.1'

In [ ]:
data_store = DataStore.from_dir('$GAMMAPY_DATA/hess-dl3-dr1')

In [ ]:
data_store.info()

## Run Selection

In [ ]:
source1 = SkyCoord(329.72*u.deg, -30.23*u.deg, frame='icrs') # PKS 2155−304

source2 = SkyCoord(228.53*u.deg, -59.16*u.deg, frame='icrs') # MSH 15−52

source3 = SkyCoord(258.35*u.deg, -39.77*u.deg, frame='icrs') # RX J1713.7−3946

Choose one of the two sources for your analysis in the next cell.

In [ ]:
## your code here

source_pos = ...


In [ ]:
print(source_pos)

Source position is set.

**<div style='text-align: right;'>[1 point]</div>**

In [ ]:
selectradius = 2.5*u.deg

In [ ]:
conesearch = data_store.obs_table.select_sky_circle(source_pos, selectradius)

In [ ]:
runlist = conesearch['OBS_ID'].value

In [ ]:
observations = data_store.get_observations(runlist)

In [ ]:
observations.ids

## Instructions

Now it is your turn. You will copy the relevant cells from the other notebooks and execute the cells. Some modifications and checks will be needed, your results may look much different than what was presented in the lectures.

## Sky Map

### Geometries

First you will need to set up the geometry, which is mainly the binning in spatial dimensions and energy. Create a MapGeom and a energy axis. You can copy that from the lecture demonstration.

In [ ]:
## your code here

map_energy_axis = ...

map_geom = ...

In [ ]:
print(map_geom)

Geometry with energy axis is set.

**<div style='text-align: right;'>[2 points]</div>**

In [ ]:
#We will also need an axis for true energy:

energy_axis_true = MapAxis.from_energy_bounds(0.08*u.TeV, 80*u.TeV,
                                              nbin=8,
                                              per_decade=True,
                                              name='energy_true')

### Excusion Mask

The exclusion mask flags regions that should be excluded from the background estimation. At least your object of interest must be contained in the exclusion mask. Make sure that the size is sufficient! If you see other objects in your sky map then they need to be added as well.

In [ ]:
## your code here

exclusion_regions = [...]

In [ ]:
exclusion_mask = map_geom.to_image().region_mask(exclusion_regions, inside = False) 

In [ ]:
exclusion_mask.plot(add_cbar = True)

Exclusion mask is set.

**<div style='text-align: right;'>[1 point]</div>**

### Create a MapDataset

Set up all the makers and datasets that are needed to create a MapDataset. Execute the loop and produce a stacked MapDataset, called ```map_stacked```.

In [ ]:
map_empty = MapDataset.create(map_geom, 
                              name='empty', 
                              energy_axis_true=energy_axis_true
                             )

Create a MapDatasetMaker

In [ ]:
## your code here

map_maker = ...


In [ ]:
print(map_maker)

Maker created.

**<div style='text-align: right;'>[2 points]</div>**

Create a SafeMaskMaker. You can use the same values as shown in the lectures.

In [ ]:
offset_max = 2.5*u.deg
aeff_max = 10

safe_mask_maker = SafeMaskMaker(methods=['offset-max', 'aeff-max'], 
                                offset_max = offset_max,
                                aeff_percent = aeff_max
                               )

Maker created.

**<div style='text-align: right;'>[3 points]</div>**

Create a RingBackgroundMaker. Do not foreget to use the exclusion mask. The inner radius should be at leat is big as your exclusion radius.

In [ ]:
ring_bkg_maker = RingBackgroundMaker(exclusion_mask=exclusion_mask,
                                     r_in=exclusion_radius,
                                     width=0.4*u.deg)

Maker created.

**<div style='text-align: right;'>[3 points]</div>**

Now we run a DatasetsMaker to use all the makers in the right order.

In [ ]:
chainmaker_map = DatasetsMaker(makers = [map_maker, safe_mask_maker, ring_bkg_maker],
                               stack_datasets = True,
                               n_jobs = 4, parallel_backend = 'multiprocessing'
                               )

map_datasets = chainmaker_map.run(dataset = map_empty.copy(name = 'stacked'),
                                  observations = observations)

map_stacked = map_datasets[0]

In [ ]:
map_stacked.peek()

### Create an Excess and Significance Map

Use the ExcessMapEstimator to obtain a significance map, called ```significance_map```. Plot the significance map with a colour bar and a colour map of your choice.  

In [ ]:
## your code here

significance_map = ...

In [ ]:
significance_map.plot(add_cbar = True,
                      cmap = 'plasma'
                     )

Significance map created.

**<div style='text-align: right;'>[3 points]</div>**

### Background Significance Distribution

Now check the distribution of your significances in the background bins. It should follow a Gaussian distribution. You may need to increase the size of your exclusion regions, or add another exclusion region.

Let's first check the existing regions:

In [ ]:
ax = significance_map.plot(add_cbar = True,
                           cmap = 'plasma'
                           )

for e in exclusion_regions :
    e.to_pixel(ax.wcs).plot(ax = ax)

Do you see significant emission outside the circle(s)? Then you should increase the size of the exclusion regions and run the code from the beginning.

In [ ]:
sig_temp = significance_map.reduce_over_axes()
ex_temp = exclusion_mask.reproject_to_geom(sig_temp.geom)
ex_temp &= map_stacked.mask_safe_image

sigmapdata_off = sig_temp.data[ex_temp.data.astype(bool)]

sig_hist = plt.hist(significance_map.data.ravel(),
                    bins = 50,
                    density = True,
                    color = 'tab:blue',
                    label = 'all bins'
                   )

sig_hist_off = plt.hist(sigmapdata_off,
                        bins = sig_hist[1],
                        density = True,
                        alpha = 0.5,
                        color = 'tab:red',
                        label = 'off bins'
                       )

plt.yscale('log')

plt.legend()

plt.xlabel('significance')

Evaluate the last plot. Maybe you need to reconsider your exclusion regions. You can also fit ```sigmapdata_off``` with a Gaussian. 

Background distribution is well normalised.

**<div style='text-align: right;'>[5 points]</div>**

### Result

Do not move on before you have all results in this step correct. You will need later the significance map, the exclusion mask, and the MapDataset.

## Spectrum

In this step, you will produce a SpectrumDataset, fit the spectrum, test for a cut-off, and produce flux points. Along the way, you will evaluate the significance of the detected emission. 

### On Region 

Create an on-region that contains all emissions. It should be smaller than the exclusion region and not much larger than the emission seen in the significance map.

In [ ]:
## your code here

on_region_radius = Angle(0.49*u.deg)

on_region = CircleSkyRegion(center=source_pos, radius=on_region_radius)

In [ ]:
zoomed_significance_map = significance_map.cutout(source_pos, 2*u.deg)

ax = zoomed_significance_map.plot(add_cbar = True, cmap = 'plasma')

on_region.to_pixel(ax.wcs).plot(color = 'white', ax = ax)

for e in exclusion_regions :
    e.to_pixel(ax.wcs).plot(ax = ax)

Well selected on region.

**<div style='text-align: right;'>[3 points]</div>**

### Energy Axis

Create an energy axis which has more bins than what you have used for the maps.

In [ ]:
## your code here

spectrum_energy_axis = ...


In [ ]:
spectrum_geom = RegionGeom.create(region=on_region, 
                                  axes=[spectrum_energy_axis]
                                 )

In [ ]:
spectrum_geom

In [ ]:
spectrum_empty =  SpectrumDataset.create(geom=spectrum_geom, 
                                         energy_axis_true=energy_axis_true
                                        )

### Create a SpectrumDataset



In [ ]:
spectrum_maker = SpectrumDatasetMaker()

Create a ReflectedRegionsBackgroundMaker. Do not forget to use the exclusion mask that you have defined earlier.

In [ ]:
## your code here

# reflected_bkg_maker =  ...

reflected_bkg_maker = ReflectedRegionsBackgroundMaker(exclusion_mask=exclusion_mask)

Maker created.

**<div style='text-align: right;'>[3 points]</div>**

In [ ]:
chainmaker_spectrum = DatasetsMaker(makers = [spectrum_maker,
                                              safe_mask_maker,
                                              reflected_bkg_maker],
                               stack_datasets = False,
                               n_jobs = 4, parallel_backend = 'multiprocessing'
                               )

spectrum_datasets = chainmaker_spectrum.run(
    dataset = spectrum_empty.copy(name = 'stacked'), ## this will not be used
    observations = observations)

In [ ]:
ax = significance_map.plot(cmap = 'plasma', add_cbar = True)

on_region.to_pixel(ax.wcs).plot(ax = ax, color = 'white')

plot_spectrum_datasets_off_regions(ax=ax, datasets=spectrum_datasets)

#plt.savefig('ReflectedRegions.svg')

### Evaluate the significance

Create the cumulative info table and get the excess and significance of your source from the last line.

In [ ]:
## your code here

excess = ...
significance = ...


In [ ]:
print(f'excess = {excess:.1f}, significance = {significance:.1f} sigma')

Values correctly reported.

**<div style='text-align: right;'>[2 points]</div>**

### Spectrum Fit
 

We will use a stacked data set from now on:

In [ ]:
spectrum_stacked = spectrum_datasets.stack_reduce()

Define two spectral models, one simple power law and one power law with exponential cut-off.

In [ ]:
## your code here

spectral_PL = ...
spectral_explPL = ...


In [ ]:
print(spectral_PL)

In [ ]:
print(spectral_explPL)

Models correctly initialised.

**<div style='text-align: right;'>[2 points]</div>**

#### Power Law

First, we create a sky model:

In [ ]:
model_PL = SkyModel(spectral_model=spectral_PL, name = 'PL')

Then we assign the model to the data set:

In [ ]:
spectrum_stacked.models = model_PL

Then we fit the model:

In [ ]:
fit.run(spectrum_stacked)

Make sure that the optimisation terminated succesfully. If not you can try to run the fit again, or to use different default parameters.

Finally, we get the test statistics:

In [ ]:
TS_PL = spectrum_stacked.stat_sum()

In [ ]:
print(TS_PL)

In [ ]:
spectrum_stacked.plot_fit()

In [ ]:
spectral_PL

#### Exponential Cut-Off Power Law

Now you do the same with the exponential cut-off.

In [ ]:
## your code here

TS_expPL = ...


Model correctly fit to the data.

**<div style='text-align: right;'>[6 points]</div>**

In [ ]:
print(TS_expPL)

TS obtained.

**<div style='text-align: right;'>[2 points]</div>**

In [ ]:
spectrum_stacked.plot_fit()

### Model Comparison

Use the TS values obtained from your fit and Wilks' theorem to decide which model will be kept. You accept the alternative model if it is preferred by 3 sigma, corresponding to a $\chi^2$ probability of $2.7 \times 10^{-3}$. Assign your chosen model (either ```spectral_PL``` or ```spectral_expPL```) to ```best_model```.

In [ ]:
## your code here

best_model = ....


In [ ]:
print(f'Delta TS = {TS_PL:.1f} - {TS_expPL:.1f} = {TS:.1f}')

In [ ]:
print(best_model)

Best model identified.

**<div style='text-align: right;'>[3 points]</div>**

### Flux Points

We assign the best-fit model to our data set:

In [ ]:
spectrum_stacked.models = SkyModel(spectral_model=best_model)

In [ ]:
spectrum_stacked.plot_fit()

Create flux points from your data set using your best-fit model. You are free to change the binning. Create a FluxPointsDataset with your flux points and your model.

In [ ]:
## your code here

flux_points_dataset = ...


In [ ]:
flux_points_dataset.plot_fit()

Model and points are plotted.

**<div style='text-align: right;'>[3 points]</div>**

## Sanity Check

Before you submit your work you should make a few checks that everything works fine.

1. Save your notebook as a PDF or HTML (File->Save and Export Notebook As->PDF). This document will help you debug in the next step.
1. Restart the kernel and rerun the entire notebook (Kernel->Restart & Run All). This will delete all variables (but not your code) and rerun the notebook in one go. If this does not go through to the end (i.e. you do not see the output of the next cell), then you have to fix it. You will see at which cell the run stopped. A common mistake is using a variable that is defined only at a later stage.
1. You think you fixed everything? Redo step 2 (Kernel->Restart & Run All)

In [ ]:
print('a\bYa\boa\bua\b a\baa\bra\bea\b a\bra\bea\baa\bda\bya\b a\bta\boa\b a\bsa\bua\bba\bma\bia\bta\b.a\b')

Do you see the output of the last cell without explicitly running it? Then the notebook goes through with one kernel restart. You can proceed to submission.
If you do not see the output? Go back to step 2 above.

The jupyter notebook goes through all cells with one Kernel Restart & Run all.

**<div style='text-align: right;'>[1 point]</div>**

## Submission

You have to download and submit 2 files: the jupyter notebook and the PDF or HTML.

Congratulations. You have successfully completed your TeV Data Analysis project.